# sparse_knn — generation and scoring

---
## 1 — Host and working tree

In [1]:
!nvidia-smi --query-gpu=name,memory.total,memory.used,driver_version --format=csv

name, memory.total [MiB], memory.used [MiB], driver_version
NVIDIA GeForce RTX 4090, 24564 MiB, 1 MiB, 590.48.01


In [2]:
from pathlib import Path

if not Path('manage.py').exists():
    if not Path('Style-Aware-MT/manage.py').exists():
        !git clone https://github.com/prnamhr/Style-Aware-MT.git
    %cd Style-Aware-MT
!git pull --ff-only
!git rev-parse --short HEAD

/workspace/Style-Aware-MT/notebooks/Style-Aware-MT
Already up to date.
c8b012b


In [8]:
%pip install -r requirements.txt

  Using cached urllib3-2.7.0-py3-none-any.whl.metadata (6.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.9/588.9 kB 11.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 42.7 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.3/532.3 MB 70.8 MB/s  0:00:07m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 65.9 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.4/556.4 kB 88.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 98.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 115.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 923.9/923.9 kB 109.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 105.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 73.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 117.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 621.4/621.4 

In [9]:
# The pipeline is text-only and these three ship against a torch the pins contradict.
%pip uninstall -q -y torchvision torchaudio torchcodec

Note: you may need to restart the kernel to use updated packages.


In [3]:
import torch

cap = torch.cuda.get_device_capability(0)
print(f'{torch.cuda.get_device_name(0)}  sm_{cap[0]}{cap[1]}',
      f'torch {torch.__version__} / cuda {torch.version.cuda}')
assert torch.cuda.is_bf16_supported(), 'bf16 unsupported; the frozen base is not quantized'

NVIDIA GeForce RTX 4090  sm_89 torch 2.12.0+cu130 / cuda 13.0


---
## 2 — Run parameters and the matched-contrast gate

In [4]:
import hashlib
import json
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import yaml

PY = sys.executable
COMET_PY = '.venv-comet/bin/python'
SPLIT = 'val'
ARM, BASELINE = 'sparse_knn', 'knn_fewshot'
CONFIG, BASE_CONFIG = Path('configs/sparse_knn.yaml'), Path('configs/base_qwen.yaml')
OUT = Path('outputs')

DIAG = Path(f'results/sparse_selection_{SPLIT}.json')
ROUTING = Path(f'results/sparse_routing_{SPLIT}.json')

N_BOOT, N_STYLO, ALPHA, SEED = 10000, 2000, 0.05, 42

CFG = yaml.safe_load(CONFIG.read_text(encoding='utf-8'))
RETR, SPA, RAR, PROMPT = CFG['retrieval'], CFG['sparse'], CFG['rarity'], CFG['prompt']
RARITY_PATH = Path(RAR['out'])
print(f"{ARM} against {BASELINE}: k={RETR['k']}, up to m={SPA['m']} rare, "
      f"{RAR['freeze_n']} rarest terms at df >= {RAR['min_df']}")

sparse_knn against knn_fewshot: k=8, up to m=4 rare, 500 rarest terms at df >= 40


In [5]:
BASE = yaml.safe_load(BASE_CONFIG.read_text(encoding='utf-8'))
for block in ('generator', 'prompt', 'retrieval', 'data', 'output'):
    assert CFG[block] == BASE[block], f'{block} differs from {BASE_CONFIG}: {CFG[block]}'
assert set(CFG) - set(BASE) == {'rarity', 'sparse'}, sorted(set(CFG) - set(BASE))
assert set(BASE) - set(CFG) == {'afsp'}, sorted(set(BASE) - set(CFG))
print(f'{CONFIG.name} differs from {BASE_CONFIG.name} in the selection blocks only')

sparse_knn.yaml differs from base_qwen.yaml in the selection blocks only


In [6]:
VAL = [json.loads(x) for x in Path(CFG['data']['eval_file']).open(encoding='utf-8') if x.strip()]
SRC = [r['input'] for r in VAL]

BASE_ROWS = [json.loads(x) for x in (OUT / f'{BASELINE}_{SPLIT}.jsonl').open(encoding='utf-8')
             if x.strip()]
assert len(BASE_ROWS) == len(VAL), f'{BASELINE}: {len(BASE_ROWS)} rows, expected {len(VAL)}'
assert [r['input'] for r in BASE_ROWS] == SRC, f'{BASELINE} is not aligned to {SPLIT}.jsonl'
assert all(r['model'] == CFG['generator']['model'] for r in BASE_ROWS), 'a different base model'
print(f'{len(VAL)} {SPLIT} segments; {BASELINE} present and aligned on {BASE_ROWS[0]["model"]}')

1323 val segments; knn_fewshot present and aligned on Qwen/Qwen2.5-7B-Instruct


---
## 3 — The rarity list and the index

In [7]:
RARITY = json.loads(RARITY_PATH.read_text(encoding='utf-8'))
RARITY_SHA = hashlib.sha256(RARITY_PATH.read_bytes()).hexdigest()

for key in ('min_df', 'freeze_n', 'zwnj'):
    assert RARITY['config'][key] == RAR[key], (key, RARITY['config'][key], RAR[key])
assert len(RARITY['terms']) == RAR['freeze_n'], f"{len(RARITY['terms'])} terms on the list"
assert RAR['min_df'] <= RARITY['df_observed'][0], RARITY['df_observed']
# The list is written rarest first, so df must not fall as the rank rises.
DFS = [df for _t, df, _tf in RARITY['terms']]
assert DFS == sorted(DFS), 'the list is not ordered from rarest to less rare'
print(f"{RARITY['n_frozen']} frozen terms of {RARITY['n_eligible']} eligible, "
      f"realized df {RARITY['df_observed']}, {RARITY['selected_frac']:.1%} of the vocabulary")
print(f'sha256 {RARITY_SHA[:16]}...')

500 frozen terms of 525 eligible, realized df [40, 410], 2.2% of the vocabulary
sha256 8fa5b0b2c0dbf344...


In [8]:
# data/knn_index is git-ignored, so a fresh session rebuilds it. It must be the same
# index knn_fewshot retrieved from, hence base_qwen.yaml rather than the arm's config.
INDEX = Path(RETR['index_dir'])
INDEX_FILES = ('embeddings.npy', 'pairs.jsonl', 'meta.json')
if not all((INDEX / f).exists() for f in INDEX_FILES):
    !python3 manage.py build_index --config configs/base_qwen.yaml

meta = json.loads((INDEX / 'meta.json').read_text(encoding='utf-8'))
assert meta['embed_model'] == RETR['embed_model'], meta
INDEX_SHA = {f: hashlib.sha256((INDEX / f).read_bytes()).hexdigest()[:16] for f in INDEX_FILES}
print(f"{meta['n_passages']} pool passages on {meta['embed_model']}, dim {meta['dim']}")
print(json.dumps(INDEX_SHA, indent=2))

10860 pool passages on intfloat/multilingual-e5-large-instruct, dim 1024
{
  "embeddings.npy": "9c282c8042ab740e",
  "pairs.jsonl": "c48f429809435593",
  "meta.json": "b028a2a81f9f0eed"
}


---
## 4 — Routing, recorded per segment


In [9]:
from src.retrieval.rarity import load_irregular
from src.retrieval.retrieve import RetrievalIndex
from src.retrieval.sparse import SparseRetriever

index = RetrievalIndex(RETR['index_dir'], embed_model=RETR['embed_model'])
retriever = SparseRetriever(
    index, load_irregular(str(RARITY_PATH)), index, zwnj=RAR['zwnj'], m=SPA['m'],
)
SELECTED, TRACES = retriever.select_with_trace(SRC, k=RETR['k'])
print(f'{len(TRACES)} traces, {len(SELECTED[0])} exemplars per prompt')

Loading intfloat/multilingual-e5-large-instruct on device: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/128 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/140k [00:00<?, ?B/s]

sentence_xlm-roberta_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

1323 traces, 8 exemplars per prompt


In [10]:
SLOTS = min(SPA['m'], RETR['k'])
HIST = {str(v): sum(t['n_sparse'] == v for t in TRACES) for v in range(SLOTS + 1)}
ROUTES = {r: sum(t['route'] == r for t in TRACES) for r in ('full', 'partial', 'dense')}

diag = json.loads(DIAG.read_text(encoding='utf-8'))
assert diag['config']['index_dir'] == RETR['index_dir'], diag['config']
assert HIST == diag['n_sparse']['histogram'], f'{HIST} against {diag["n_sparse"]["histogram"]}'
assert ROUTES == diag['routes'], f'{ROUTES} against {diag["routes"]}'

ROUTING.write_text(json.dumps({
    'split': SPLIT,
    'index_dir': RETR['index_dir'],
    'rarity_sha256': RARITY_SHA,
    'k': RETR['k'],
    'm': SPA['m'],
    'routes': ROUTES,
    'n_sparse_histogram': HIST,
    'segments': [{'route': t['route'], 'n_sparse': t['n_sparse'], 'n_knn': t['n_knn'],
                  'n_query_terms': t['n_query_terms'], 'n_targeted': t['n_targeted'],
                  'served_terms': t['served_terms']}
                 for t in TRACES],
}, ensure_ascii=False, indent=2), encoding='utf-8')

routed = len(TRACES) - ROUTES['dense']
print(f'routes {ROUTES}  ({routed / len(TRACES):.1%} routed)')
print(f'rare slots filled {HIST}, mean {np.mean([t["n_sparse"] for t in TRACES]):.3f}')
print(f'matches {DIAG}; wrote {ROUTING}')

routes {'full': 563, 'partial': 639, 'dense': 121}  (90.9% routed)
rare slots filled {'0': 121, '1': 208, '2': 252, '3': 179, '4': 563}, mean 2.646
matches results/sparse_selection_val.json; wrote results/sparse_routing_val.json


---
## 5 — Stage B: the assembled prompt

In [11]:
from src.infer.run import _load_configured_glossary, build_fewshot_user, order_exemplars

K = RETR['k']
STYLE = Path(PROMPT['style_instruction_file']).read_text(encoding='utf-8')
GLOSSARY = _load_configured_glossary(CFG)
ORDERED = [order_exemplars(ex, PROMPT['ordering']) for ex in SELECTED]
PROMPTS = [build_fewshot_user(s, ex, GLOSSARY) for s, ex in zip(SRC, ORDERED)]


def terms_of(text):
    """The listed rare terms a string carries, as the channel itself tokenises it."""
    return {retriever.terms[c] for c in retriever.query_terms(text)}


for i, (ex, t) in enumerate(zip(ORDERED, TRACES)):
    keys = [(e['input'], e['output']) for e in ex]
    assert len(keys) == K, f'segment {i}: {len(keys)} exemplars, expected {K}'
    assert len(set(keys)) == len(keys), f'segment {i}: an exemplar repeats across the channels'
    assert SRC[i] not in {e['input'] for e in ex}, f'segment {i}: the query is its own exemplar'
    assert [e['input'] for e in ex] == [index.pairs[r]['input'] for r in t['final_rows'][::-1]], (
        f'segment {i}: prompt order is not the reversed cosine ranking')
    # One pick per served term, in the same order, each carrying the term it answers.
    assert len(t['sparse_rows']) == len(t['served_terms']), f'segment {i}: picks without a term'
    for r, term in zip(t['sparse_rows'], t['served_terms']):
        assert term in terms_of(index.pairs[r]['input']), (
            f'segment {i}: rare pick {r} does not carry {term}')

print(f'{len(PROMPTS)} prompts: {K} distinct exemplars each, cosine-ranked order, '
      f'every rare pick carrying the term it was fetched for')
print(f'median prompt {int(np.median([len(p) for p in PROMPTS]))} chars, '
      f'longest {max(len(p) for p in PROMPTS)}')

1323 prompts: 8 distinct exemplars each, cosine-ranked order, every rare pick carrying the term it was fetched for
median prompt 5429 chars, longest 9895


In [12]:
RAR_LEN, COS_LEN = [], []
for t in TRACES:
    rows = set(t['sparse_rows'])
    for r in t['final_rows']:
        (RAR_LEN if r in rows else COS_LEN).append(len(index.pairs[r]['input']))

BASE_SEL = index.retrieve(SRC, k=K)
BASE_CHARS = sum(len(e['input']) for row in BASE_SEL for e in row) / len(TRACES)
ARM_CHARS = (sum(RAR_LEN) + sum(COS_LEN)) / len(TRACES)

for name, v in (('query', [len(x) for x in SRC]), ('rarity channel', RAR_LEN),
                ('cosine fill', COS_LEN)):
    a = np.array(v)
    print(f'{name:16s} n={len(a):6d}  mean {a.mean():6.1f}  median {np.median(a):6.1f} chars')
print(f'exemplar chars per prompt: {BASELINE} {BASE_CHARS:.0f} -> {ARM} {ARM_CHARS:.0f} '
      f'({ARM_CHARS / BASE_CHARS - 1:+.1%})')

query            n=  1323  mean   78.2  median   63.0 chars
rarity channel   n=  3501  mean  197.2  median  175.0 chars
cosine fill      n=  7083  mean  198.8  median  181.0 chars
exemplar chars per prompt: knn_fewshot 1603 -> sparse_knn 1586 (-1.1%)


In [13]:
FILLED = np.array([t['n_sparse'] for t in TRACES])
SHOW = ([int(i) for i in np.flatnonzero(FILLED == SLOTS)[:2]]
        + [int(i) for i in np.flatnonzero((FILLED > 0) & (FILLED < SLOTS))[:2]]
        + [int(i) for i in np.flatnonzero(FILLED == 0)[:1]])
QEMB = index.encode([SRC[i] for i in SHOW])

for q, i in zip(QEMB, SHOW):
    t = TRACES[i]
    rarity = set(t['sparse_rows'])
    print(f"[{i}] {t['route']}  {t['n_sparse']} rare + {t['n_knn']} kNN")
    print('  query:', SRC[i][:100])
    print(f"  terms: {t['n_query_terms']} carried, served {t['served_terms']}")
    for pos, row in enumerate(t['final_rows'][::-1], start=1):
        e = index.pairs[row]
        hit = sorted(terms_of(e['input']) & set(t['query_terms']))
        shared = ('+' + ' '.join(hit[:3])) if hit else ''
        print(f"    {pos}. {'rare  ' if row in rarity else 'cosine'}  "
              f"cos {float(index.embeddings[row] @ q):.3f}  {shared:24s} {e['input'][:55]}")
    print()

[0] full  4 rare + 4 kNN
  query: جواهر الأسرار فی معارج الأسفار لمن اراد ان یتقرّب بالله المقتدر الغفّار فهنیاً للأبرار الّذین یشربون
  terms: 6 carried, served ['اراد', 'بالله', 'هذه', 'لمن']
    1. rare    cos 0.870  +اراد لمن                لو تسمع صریر القلم الأعلی و هدیر ورقآء البقآء علی افنان
    2. rare    cos 0.874  +بالله هذه               تفکّروا فی هذه الآیة ثم انصفوا بالله لعلّ تجدون لئالئ ا
    3. rare    cos 0.876  +هذه                     اسألک بضجیج المشتاقین فی هجرک و صریخ العاشقین فی بعدهم 
    4. rare    cos 0.877  +لمن                     طوبی لمن فاز بلقائک و شرب رحیق الوصال من ایادی عطائک و 
    5. cosine  cos 0.879  +الذین                   من شرب من الکأس الّتی تدور بها ید رحمتک ینقطع عن دونک و
    6. cosine  cos 0.881                           اللّهمّ انّی اسألک بالحرف الّتی اذا خرجت من فم مشیّتک م
    7. cosine  cos 0.882                           فأمطر من سحاب فیض فضلک ما تطهّر به افئدة عبادک عمّا یحج
    8. cosine  cos 0.886                           و بهدا

In [14]:
i = SHOW[0]
print(STYLE)
print('=' * 88)
print(PROMPTS[i])

You are an expert translator of Bahá'í scripture from Persian and Arabic into English.

Render the source text into English in the formal, elevated, scriptural register of Shoghi Effendi's authorized translations. Observe the following:

- Preserve the dignity and cadence of sacred prose; favour the elevated, archaic register over modern neutral English.
- Use the second-person sacred pronouns and their verb forms where the source addresses the Divine or is addressed by It: "Thou", "Thee", "Thy", "Thine", and verb endings such as "art", "hast", "dost", "doth".
- Retain formal vocatives such as "O" and honorific constructions where the source warrants them.
- Translate the full meaning faithfully; do not add commentary, explanation, transliteration, or footnotes.
- Output only the English translation, as a single continuous passage with no quotation marks, labels, or preamble.

Here are example translations in the required style:

[Terms] الله → God | قل → Say
Source: لو تسمع صریر القلم

---
## 6 — Generation

In [15]:
import getpass
import logging
import os

# HF_TOKEN only. The adapter repo is private and the base model is public; nothing in this
# notebook can spend, so a rater key present here would be a mistake, not a convenience.
if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass.getpass('HF_TOKEN: ')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
logging.getLogger('httpx').setLevel(logging.WARNING)
print('HF_TOKEN set, no rater keys present')

HF_TOKEN:  ········


HF_TOKEN set, no rater keys present


In [17]:
t0 = time.perf_counter()
r = subprocess.run([PY, 'manage.py', 'infer', '--condition', ARM, '--config', str(CONFIG)],
                   check=False)
assert r.returncode == 0, f'{ARM} exited {r.returncode}'
GEN_SECONDS = round(time.perf_counter() - t0, 1)
print(f'{GEN_SECONDS / 60:.1f} min, finished {datetime.now(timezone.utc).isoformat()}')

sparse_knn: k=8 as up to 4 rarity + cosine for 1323 ...
Loading intfloat/multilingual-e5-large-instruct on device: cuda


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 4200.35it/s]


  routes: {'full': 563, 'partial': 639, 'dense': 121}, mean rare slots filled: 2.65


Loading weights: 100%|██████████| 339/339 [00:02<00:00, 149.36it/s]


resuming sparse_knn: 38/1323 already done
Generating 1323 translations with Qwen/Qwen2.5-7B-Instruct (sparse_knn) ...
  40/1323
  45/1323
  50/1323
  55/1323
  60/1323
  65/1323
  70/1323
  75/1323
  80/1323
  85/1323
  90/1323
  95/1323
  100/1323
  105/1323
  110/1323
  115/1323
  120/1323
  125/1323
  130/1323
  135/1323
  140/1323
  145/1323
  150/1323
  155/1323
  160/1323
  165/1323
  170/1323
  175/1323
  180/1323
  185/1323
  190/1323
  195/1323
  200/1323
  205/1323
  210/1323
  215/1323
  220/1323
  225/1323
  230/1323
  235/1323
  240/1323
  245/1323
  250/1323
  255/1323
  260/1323
  265/1323
  270/1323
  275/1323
  280/1323
  285/1323
  290/1323
  295/1323
  300/1323
  305/1323
  310/1323
  315/1323
  320/1323
  325/1323
  330/1323
  335/1323
  340/1323
  345/1323
  350/1323
  355/1323
  360/1323
  365/1323
  370/1323
  375/1323
  380/1323
  385/1323
  390/1323
  395/1323
  400/1323
  405/1323
  410/1323
  415/1323
  420/1323
  425/1323
  430/1323
  435/1323
  440/1323
  4

---
## 7 — The output and its provenance

In [51]:
ARM_PATH = OUT / f'{ARM}_{SPLIT}.jsonl'
ARM_ROWS = [json.loads(x) for x in ARM_PATH.open(encoding='utf-8') if x.strip()]

assert len(ARM_ROWS) == len(VAL), f'{len(ARM_ROWS)} rows, expected {len(VAL)}'
assert [r['input'] for r in ARM_ROWS] == SRC, 'source order differs from the eval file'
assert all(r['condition'] == ARM for r in ARM_ROWS), 'mislabelled rows'
blank = [i for i, r in enumerate(ARM_ROWS) if not r['prediction'].strip()]
errored = [i for i, r in enumerate(ARM_ROWS) if 'error' in r]
assert not errored, f'{len(errored)} segments recorded an error: {errored[:5]}'
print(f'{len(ARM_ROWS)} rows, {len(blank)} blank predictions, {len(errored)} errors')

1323 rows, 0 blank predictions, 0 errors


In [52]:
USAGE = json.loads((OUT / f'{ARM}_{SPLIT}_usage.json').read_text(encoding='utf-8'))
PROV = USAGE['provenance']

assert PROV['index_dir'] == RETR['index_dir'], PROV['index_dir']
assert PROV['rarity_sha256'] == RARITY_SHA, f"{PROV['rarity_sha256']} != {RARITY_SHA}"
assert PROV['k'] == RETR['k'] and PROV['m'] == SPA['m'], PROV
assert all(PROV[key] == RAR[key] for key in ('min_df', 'freeze_n')), PROV
assert PROV['ordering'] == PROMPT['ordering'], PROV
print(json.dumps(PROV, indent=2))
print(f"{USAGE['calls']} calls, ${USAGE.get('cost_usd', 0.0):.2f} — local weights, nothing paid")

{
  "k": 8,
  "ordering": "most_similar_last",
  "index_dir": "data/knn_index",
  "m": 4,
  "min_df": 40,
  "freeze_n": 500,
  "rarity_file": "results/rarity_train.json",
  "rarity_sha256": "8fa5b0b2c0dbf3441254c1ce6ad6a0a91a50fd90583a18f1991f5fa263ca8d49"
}
1285 calls, $0.00 — local weights, nothing paid


---
## 8 — Divergence from knn_fewshot

In [53]:
ARM_PRED = [r['prediction'] for r in ARM_ROWS]
BASE_PRED = [r['prediction'] for r in BASE_ROWS]
DIFFERS = np.array([a != b for a, b in zip(ARM_PRED, BASE_PRED)])
ROUTE = np.array([t['route'] for t in TRACES])
N_SPARSE = np.array([t['n_sparse'] for t in TRACES])

share = DIFFERS.mean()
print(f'{DIFFERS.sum()}/{len(DIFFERS)} predictions differ ({share:.1%}); '
      f'routed fraction is {routed / len(TRACES):.1%}')
for r in ('full', 'partial', 'dense'):
    sel = ROUTE == r
    print(f'  {r:8s} n={sel.sum():5d}  differ {DIFFERS[sel].mean():.1%}')

994/1323 predictions differ (75.1%); routed fraction is 90.9%
  full     n=  563  differ 92.2%
  partial  n=  639  differ 67.8%
  dense    n=  121  differ 34.7%


In [54]:
dense_diff = DIFFERS[ROUTE == 'dense'].mean()
assert share >= 0.40, (
    f'only {share:.1%} of predictions differ against a {routed / len(TRACES):.1%} routed '
    f'fraction; the rarity channel is not reaching the prompt')
if not 0.60 <= share <= 0.98:
    print(f'NOTE: {share:.1%} differ, outside the 60-98% this design usually lands in. '
          f'Worth reading the per-route rows above before treating the deltas as selection.')
if dense_diff:
    print(f'WARNING: {dense_diff:.1%} of dense-routed segments differ despite an identical '
          f'prompt — greedy decoding did not reproduce across sessions, so read the deltas '
          f'below against this floor, not against zero.')
else:
    print('every dense-routed segment reproduces the baseline exactly; the deltas below '
          'carry no decode noise')

In [55]:
i = int(np.flatnonzero((ROUTE == 'full') & DIFFERS)[0])
print('SOURCE  :', SRC[i][:110])
print('terms   :', TRACES[i]['query_terms'][:8], f"served {TRACES[i]['served_terms']}")
print(f'{BASELINE:12s}:', BASE_PRED[i][:200])
print(f'{ARM:12s}:', ARM_PRED[i][:200])

SOURCE  : جواهر الأسرار فی معارج الأسفار لمن اراد ان یتقرّب بالله المقتدر الغفّار فهنیاً للأبرار الّذین یشربون من هذه ال
terms   : ['اراد', 'بالله', 'هذه', 'لمن', 'المقتدر', 'الذین'] served ['اراد', 'بالله', 'هذه', 'لمن']
knn_fewshot : Jewels of the mysteries in the ascent of journeys for him who desireth to draw nigh to God, the Almighty, the Forgiver; blessed indeed are the righteous who drink of these rivers.
sparse_knn  : O God, the pearls of mysteries lie hidden in the stations of journeys for him who desireth to draw nigh unto Thee, the All-Powerful, the Forgiver. Verily, a blessed lot for the righteous who drink fro


---
## 9 — Stage D: scoring

In [56]:
CONDS = [BASELINE, ARM]
!{PY} manage.py eval --conditions {' '.join(CONDS)} --split {SPLIT}

condition    n     BLEU   chrF   marker_rate  ref_marker_rate
-------------------------------------------------------------
knn_fewshot  1323  13.99  39.82  1.23         0.93           
sparse_knn   1323  14.31  40.08  1.25         0.93           


In [37]:
from src.eval.quick import score

SURFACE = {c: score(c, OUT, SPLIT) for c in CONDS}
print(f"{'condition':14s} {'chrF':>8s} {'BLEU':>8s} {'markers/seg':>12s}")
for cond in CONDS:
    s = SURFACE[cond]
    print(f"{cond:14s} {s['chrF']:8.2f} {s['BLEU']:8.2f} {s['marker_rate']:12.2f}")
print(f"gold targets carry {SURFACE[BASELINE]['ref_marker_rate']:.2f} markers per segment")

condition          chrF     BLEU  markers/seg
knn_fewshot       39.82    13.99         1.23
sparse_knn        40.08    14.31         1.25
gold targets carry 0.93 markers per segment


### COMET

In [38]:
if not Path(COMET_PY).exists():
    pip = [COMET_PY, '-m', 'pip', 'install', '-q']
    subprocess.run([PY, '-m', 'venv', '.venv-comet'], check=True)
    subprocess.run([*pip, '--upgrade', 'pip'], check=True)
    subprocess.run([*pip, 'setuptools<81'], check=True)
    subprocess.run([*pip, '-r', 'requirements-comet.txt'], check=True)

COMET_PATH = f'results/comet_{SPLIT}.json'
PRIOR_COMET = set(json.loads(Path(COMET_PATH).read_text(encoding='utf-8')))
assert BASELINE in PRIOR_COMET, f'{BASELINE} is not in {COMET_PATH} to pair against'

r = subprocess.run([COMET_PY, 'manage.py', 'comet', '--conditions', ARM, '--split', SPLIT,
                    '--results_path', COMET_PATH, '--batch_size', '16'], check=False)
assert r.returncode == 0, f'comet exited {r.returncode}'

Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 1237.99it/s]
Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
Encoder model frozen.
/home/prnamhr/projects/Style-Aware-MT/.venv-comet/lib/python3.11/site-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and ver

sparse_knn       COMET 0.6846  (n=1323)
preserved 14 condition(s) not scored here: afsp_full, afsp_full_casefix, afsp_margin, commercial_haiku, knn_fewshot, peft, peft_afsp, peft_afsp_casefix, peft_knn, random_fewshot, rlsf_w3_0.0, rlsf_w3_2.0, rlsf_w3_6.0, zeroshot
Wrote results/comet_val.json


In [39]:
COMET = json.loads(Path(COMET_PATH).read_text(encoding='utf-8'))
assert PRIOR_COMET <= set(COMET), f'lost from {COMET_PATH}: {sorted(PRIOR_COMET - set(COMET))}'
ref, arm = COMET[BASELINE], COMET[ARM]
assert arm['n'] == len(VAL), arm['n']
assert arm['model'] == ref['model'], (arm['model'], ref['model'])
assert arm['sources'] == ref['sources'], 'the two conditions are not paired segment for segment'
print(f"{ref['model']}: {BASELINE} {ref['system']:.4f} -> {ARM} {arm['system']:.4f}")

Unbabel/wmt22-comet-da: knn_fewshot 0.6839 -> sparse_knn 0.6846


### Register fit

In [40]:
!{PY} manage.py stylometrics --conditions {' '.join(CONDS)} --split {SPLIT} --targets-split train

label         n      lex_density  lex_density_sd  ttr     ttr_sd  root_ttr  root_ttr_sd  sent_len_mean  sent_len_mean_sd  sent_len_var  sent_len_var_sd  marker_rate  marker_rate_sd  stylo_dist
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
target:train  10860  0.4344       0.1101          0.854   0.1085  4.0437    1.0426       24.7388        16.6125           6.6894        58.4011          0.0573       0.0782          0.0       
knn_fewshot   1323   0.4034       0.1042          0.8367  0.1181  3.8663    0.9455       24.1354        15.6066           5.3587        51.9888          0.0559       0.0778          0.3659    
sparse_knn    1323   0.4027       0.102           0.8357  0.1199  3.8645    0.9432       24.1243        15.8286           4.8616        42.7644          0.0571       0.0794          0.375     


In [41]:
STYLO_PATH = f'results/stylometrics_ci_{ARM}_{SPLIT}.json'
!{PY} manage.py stylometrics_ci --split {SPLIT} --conditions {' '.join(CONDS)} \
    --n_resamples {N_STYLO} --alpha {ALPHA} --seed {SEED} --results_path {STYLO_PATH}


Register fit of the main conditions  (split=val, n=1323 segments, resamples=2000, seed=42)
stylo_dist = standardized distance to the target-register centroid; lower is better.

rank  condition    stylo_dist  ci95              P(this rank)  modal rank  mean rank
------------------------------------------------------------------------------------
1     knn_fewshot  0.3659      [0.3214, 0.4177]  0.754         1 (0.754)   1.25     
2     sparse_knn   0.3750      [0.3291, 0.4263]  0.754         2 (0.754)   1.75     

Signed z per register feature (95% CI; 0 = on target)
condition    lex_density              ttr                      root_ttr                 marker_rate           
--------------------------------------------------------------------------------------------------------------
knn_fewshot  -0.281 [-0.333, -0.228]  -0.159 [-0.218, -0.104]  -0.170 [-0.220, -0.121]  -0.018 [-0.073, 0.036]
sparse_knn   -0.288 [-0.339, -0.237]  -0.168 [-0.228, -0.110]  -0.172 [-0.220, -0.123]  -0.003

In [42]:
import math

STYLO = json.loads(Path(STYLO_PATH).read_text(encoding='utf-8'))
old = json.loads(Path(f'results/stylometrics_ci_{SPLIT}.json').read_text(encoding='utf-8'))
a, b = STYLO['cells'][BASELINE], old['cells'][BASELINE]
assert a['z'] == b['z'], f'{BASELINE} z moved between passes: {a["z"]} vs {b["z"]}'
assert math.isclose(a['stylo_dist'], b['stylo_dist'], rel_tol=1e-9), (
    f'{BASELINE} stylo_dist {a["stylo_dist"]!r} against committed {b["stylo_dist"]!r}')
print(f"{BASELINE} reproduces the committed row: stylo_dist {a['stylo_dist']:.4f}")
for cond in CONDS:
    print(f"  {cond:14s} stylo_dist {STYLO['cells'][cond]['stylo_dist']:.4f}")

knn_fewshot reproduces the committed row: stylo_dist 0.3659
  knn_fewshot    stylo_dist 0.3659
  sparse_knn     stylo_dist 0.3750


---
## 10 — Phi (paid)

In [45]:
JUDGE_CFG = 'configs/judge_eval.yaml'
JUDGE_RESULTS = f'results/judge_{SPLIT}.json'
JUDGE_USAGE = f'results/judge_{SPLIT}_usage.json'
JUDGE_CI_PATH = f'results/judge_ci_{ARM}_{SPLIT}.json'

PRIOR_JUDGE = json.loads(Path(JUDGE_RESULTS).read_text(encoding='utf-8'))
PRIOR_USAGE = json.loads(Path(JUDGE_USAGE).read_text(encoding='utf-8'))
PRIOR_SPEND, PRIOR_CALLS = (PRIOR_USAGE['cumulative'][k] for k in ('cost_usd', 'calls'))

assert BASELINE in PRIOR_JUDGE, f'{BASELINE} has no Phi to compare against'
BUY = [c for c in CONDS if c not in PRIOR_JUDGE]
PER_CALL = PRIOR_USAGE['cumulative']['cost_usd'] / PRIOR_USAGE['cumulative']['calls']
N_CALLS = len(VAL) * len(BUY)
PROJECTED = PER_CALL * N_CALLS
print(f"buying Phi for {BUY or 'nothing'}: {N_CALLS} calls at ${PER_CALL:.5f} "
      f"= ${PROJECTED:.2f} projected (cumulative judge spend ${PRIOR_SPEND:.2f})")

buying Phi for nothing: 0 calls at $0.00102 = $0.00 projected (cumulative judge spend $8.09)


In [46]:
# Left False so a top-to-bottom re-run cannot authorise itself.
SPEND_OK = True
BUDGET_USD = 1.80
N_PILOT = 25

assert PROJECTED <= BUDGET_USD, f'projection ${PROJECTED:.2f} exceeds the ${BUDGET_USD:.2f} cap'
print(f'authorised {SPEND_OK}   cap ${BUDGET_USD:.2f}   pilot {N_PILOT} x {len(BUY)} '
      f'(${PER_CALL * N_PILOT * len(BUY):.3f})')

authorised True   cap $1.80   pilot 25 x 0 ($0.000)


In [47]:
if not BUY:
    print('nothing to buy: every condition already carries Phi')
else:
    assert SPEND_OK, 'set SPEND_OK = True in the cell above to authorise the pilot'
    r = subprocess.run([PY, 'manage.py', 'judge', '--conditions', *BUY, '--split', SPLIT,
                        '--config', JUDGE_CFG, '--limit', str(N_PILOT)], check=False)
    assert r.returncode == 0, f'pilot exited {r.returncode}'

nothing to buy: every condition already carries Phi


In [48]:
_u = json.loads(Path(JUDGE_USAGE).read_text(encoding='utf-8'))
if _u['conditions'] != sorted(BUY) or _u['limit'] != N_PILOT:
    REVISED = PROJECTED
    print(f"{JUDGE_USAGE} holds {_u['conditions']} at limit {_u['limit']}, not this pilot: "
          f'${PROJECTED:.2f} stands')
else:
    pilot = _u['session']
    REVISED = pilot['cost_usd'] / pilot['calls'] * N_CALLS
    print(f"pilot {pilot['calls']} calls at ${pilot['cost_usd'] / pilot['calls']:.5f} "
          f'-> ${REVISED:.2f} for the full pass')
assert REVISED <= BUDGET_USD, f'revised ${REVISED:.2f} exceeds the ${BUDGET_USD:.2f} cap'

results/judge_val_usage.json holds ['sparse_knn'] at limit None, not this pilot: $0.00 stands


In [49]:
# The client is built on first call, so re-running over a complete cache makes no request.
if not BUY:
    print('nothing to buy; the results file already carries every condition')
else:
    assert SPEND_OK, 'set SPEND_OK = True to authorise the full pass'
    r = subprocess.run([PY, 'manage.py', 'judge', '--conditions', *BUY, '--split', SPLIT,
                        '--config', JUDGE_CFG], check=False)
    assert r.returncode == 0, f'judge exited {r.returncode}'

nothing to buy; the results file already carries every condition


In [50]:
JUDGE = json.loads(Path(JUDGE_RESULTS).read_text(encoding='utf-8'))
HAVE_PHI = ARM in JUDGE
if HAVE_PHI:
    r = subprocess.run([PY, 'manage.py', 'judge_ci', '--split', SPLIT, '--conditions', *CONDS,
                        '--n_resamples', str(N_BOOT), '--alpha', str(ALPHA), '--seed', str(SEED),
                        '--results_path', JUDGE_CI_PATH], check=False)
    assert r.returncode == 0, f'judge_ci exited {r.returncode}'
else:
    print(f'{ARM} carries no Phi; sections 10 and 11 report adequacy and register only')


Judge register fidelity Phi by condition  (split=val, n=1323 segments, resamples=10000, seed=42)
judge: claude-haiku-4-5  [tag (none)]
Phi = mean 1-5 rubric rating against the authorized reference; higher is better.

rank  condition    class  n     Phi     ci95              sd     P(this rank)  modal rank  mean rank
----------------------------------------------------------------------------------------------------
1     sparse_knn   study  1323  2.8125  [2.7649, 2.8625]  0.915  0.998         1 (0.998)   1.00     
2     knn_fewshot  study  1322  2.7481  [2.6982, 2.7995]  0.928  0.998         2 (0.998)   2.00     

Score distribution over the rubric (share of parsed segments)
condition    coverage  =1     =2     =3     =4     =5   
--------------------------------------------------------
sparse_knn   1.0000    0.067  0.320  0.357  0.246  0.010
knn_fewshot  0.9992    0.076  0.352  0.326  0.239  0.007

Adjacent ranks, paired bootstrap on the shared resamples  (a - b, 95% CI)
comparison  

In [58]:
import json
seg = json.load(open('results/sparse_routing_val.json'))['segments']
route = [s['route'] for s in seg]
new  = preds('outputs/sparse_knn_val.jsonl')
base = preds('outputs/knn_fewshot_val.jsonl')
d = [i for i,r in enumerate(route) if r=='dense']
diff = [i for i in d if new[i]!=base[i]]
print(f'{len(diff)}/{len(d)} dense-routed segments differ')

42/121 dense-routed segments differ


In [60]:
lost = sorted(set(PRIOR_JUDGE) - set(JUDGE))
assert not lost, f'lost from {JUDGE_RESULTS}: {lost}'
if HAVE_PHI:
    assert JUDGE[ARM]['model'] == JUDGE[BASELINE]['model'], 'two raters, not one'

usage = json.loads(Path(JUDGE_USAGE).read_text(encoding='utf-8'))
assert usage['priced'], 'the judge model has no pricing table; cost_usd is a floor, not a bill'
SPENT = usage['cumulative']['cost_usd'] - PRIOR_SPEND
PAID_CALLS = usage['cumulative']['calls'] - PRIOR_CALLS
assert SPENT <= BUDGET_USD, f'${SPENT:.2f} spent against a ${BUDGET_USD:.2f} cap'

METRICS = ['chrf', 'bleu', 'comet'] + (['judge'] if HAVE_PHI else [])
print(f"{PAID_CALLS} paid calls, ${SPENT:.2f} on {usage['model']} this session; "
      f"cumulative ${usage['cumulative']['cost_usd']:.2f}")
print('reading out on', ', '.join(METRICS))

0 paid calls, $0.00 on claude-haiku-4-5 this session; cumulative $8.09
reading out on chrf, bleu, comet, judge


---
## 11 — The paired bootstrap

In [61]:
BOOT_PATHS = {}
for metric in METRICS:
    path = f'results/bootstrap_{metric}_{ARM}_{SPLIT}.json'
    r = subprocess.run([PY, 'manage.py', 'bootstrap', '--metric', metric,
                        '--conditions', *CONDS, '--split', SPLIT, '--pairs', f'{ARM}:{BASELINE}',
                        '--n_resamples', str(N_BOOT), '--alpha', str(ALPHA), '--seed', str(SEED),
                        '--out', path], check=False)
    assert r.returncode == 0, f'{metric} bootstrap exited {r.returncode}'
    BOOT_PATHS[metric] = path

wrote results/bootstrap_chrf_sparse_knn_val.json

chrf paired bootstrap  (resamples=10000, split=val)
comparison                n     diff   ci95             p       sig
-------------------------------------------------------------------
sparse_knn - knn_fewshot  1323  0.308  [-0.045, 0.655]  0.0884     

* = 95% CI excludes 0 (difference significant at α=0.05)
wrote results/bootstrap_bleu_sparse_knn_val.json

bleu paired bootstrap  (resamples=10000, split=val)
comparison                n     diff  ci95             p       sig
------------------------------------------------------------------
sparse_knn - knn_fewshot  1323  0.34  [-0.008, 0.693]  0.0544     

* = 95% CI excludes 0 (difference significant at α=0.05)
wrote results/bootstrap_comet_sparse_knn_val.json

comet paired bootstrap  (resamples=10000, split=val)
comparison                n     diff   ci95             p      sig
------------------------------------------------------------------
sparse_knn - knn_fewshot  1323  0.001

---
## 12 — Read-out by dose

In [62]:
from src.eval.bootstrap import _load_segment_scores, paired_bootstrap

ROUTE_JSON = json.loads(ROUTING.read_text(encoding='utf-8'))['segments']
assert len(ROUTE_JSON) == len(VAL), len(ROUTE_JSON)

STRATA = {
    f'full dose (n_sparse = {SLOTS})': [i for i, t in enumerate(ROUTE_JSON)
                                        if t['n_sparse'] == SLOTS],
    'all routed': [i for i, t in enumerate(ROUTE_JSON) if t['route'] != 'dense'],
    'all segments': list(range(len(VAL))),
}
# Declared at n=1,323; scaled by 1/sqrt(n) for the subsets.
FLOOR = {'judge': 0.058, 'comet': 0.005}
print({k: len(v) for k, v in STRATA.items()})

{'full dose (n_sparse = 4)': 563, 'all routed': 1202, 'all segments': 1323}


In [63]:
SCORES = {}
for metric in METRICS:
    scores, sources = _load_segment_scores(metric, CONDS, OUT, SPLIT, None)
    for cond in CONDS:
        assert cond in scores, f'{metric}: {cond} has no per-segment scores'
        assert len(scores[cond]) == len(VAL), (metric, cond, len(scores[cond]))
        if sources.get(cond) is not None:
            assert sources[cond] == SRC, f'{metric}/{cond}: segment order is not the eval order'
    SCORES[metric] = scores
print('per-segment scores aligned to the eval order for', ', '.join(SCORES))

per-segment scores aligned to the eval order for chrf, bleu, comet, judge


In [64]:
PLACES = {'chrf': 2, 'bleu': 2, 'comet': 4, 'judge': 4}

def delta(metric, idx):
    a = [SCORES[metric][ARM][i] for i in idx]
    b = [SCORES[metric][BASELINE][i] for i in idx]
    return paired_bootstrap(a, b, n_resamples=N_BOOT, alpha=ALPHA, seed=SEED)

for metric in METRICS:
    p = PLACES[metric]
    print(f'\n{metric}   {ARM} - {BASELINE}')
    for label, idx in STRATA.items():
        d = delta(metric, idx)
        mark = '*' if d['significant'] else ' '
        line = (f"  {label:26s} n={d['n']:5d}  {d['diff']:+.{p}f} "
                f"[{d['ci_low']:+.{p}f}, {d['ci_high']:+.{p}f}]  p={d['p_value']:.4f} {mark}")
        if metric in FLOOR:
            f = FLOOR[metric] * (len(VAL) / d['n']) ** 0.5
            line += f"  floor {f:.{p}f}{'' if abs(d['diff']) >= f else '  (under)'}"
        print(line)


chrf   sparse_knn - knn_fewshot
  full dose (n_sparse = 4)   n=  563  +0.42 [-0.02, +0.86]  p=0.0588  
  all routed                 n= 1202  +0.40 [+0.03, +0.77]  p=0.0336 *
  all segments               n= 1323  +0.31 [-0.04, +0.65]  p=0.0884  

bleu   sparse_knn - knn_fewshot
  full dose (n_sparse = 4)   n=  563  +0.31 [-0.16, +0.80]  p=0.1964  
  all routed                 n= 1202  +0.37 [-0.01, +0.76]  p=0.0550  
  all segments               n= 1323  +0.34 [-0.01, +0.69]  p=0.0544  

comet   sparse_knn - knn_fewshot
  full dose (n_sparse = 4)   n=  563  -0.0001 [-0.0029, +0.0028]  p=0.9402    floor 0.0077  (under)
  all routed                 n= 1202  +0.0009 [-0.0016, +0.0036]  p=0.4506    floor 0.0052  (under)
  all segments               n= 1323  +0.0007 [-0.0017, +0.0031]  p=0.5670    floor 0.0050  (under)

judge   sparse_knn - knn_fewshot
  full dose (n_sparse = 4)   n=  563  +0.0568 [-0.0071, +0.1226]  p=0.0852    floor 0.0889  (under)
  all routed                 n= 1202  +0

In [65]:
# The CLI's own table, for the pair the command line names.
for metric, path in BOOT_PATHS.items():
    rec = next(r for r in json.loads(Path(path).read_text(encoding='utf-8'))['comparisons']
               if (r['a'], r['b']) == (ARM, BASELINE))
    p = PLACES[metric]
    print(f"{metric:6s} {rec['diff']:+.{p}f} [{rec['ci_low']:+.{p}f}, {rec['ci_high']:+.{p}f}] "
          f"p={rec['p_value']:.4f} n={rec['n']}")

chrf   +0.31 [-0.04, +0.65] p=0.0884 n=1323
bleu   +0.34 [-0.01, +0.69] p=0.0544 n=1323
comet  +0.0007 [-0.0017, +0.0031] p=0.5670 n=1323
judge  +0.0658 [+0.0204, +0.1104] p=0.0042 n=1322


---
## 13 — Phi_B

In [66]:
# The batch endpoint spends, so the rater key is entered here rather than beside HF_TOKEN.
if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OPENAI_API_KEY: ')
print('OPENAI_API_KEY set')

OPENAI_API_KEY set


In [67]:
JUDGE_CFG_B = 'configs/judge_eval_gpt.yaml'
TAG_B = 'gpt'
JUDGE_RESULTS_B = f'results/judge_{TAG_B}_{SPLIT}.json'
JUDGE_USAGE_B = f'results/judge_{TAG_B}_{SPLIT}_usage.json'
JUDGE_CI_PATH_B = f'results/judge_ci_{TAG_B}_{ARM}_{SPLIT}.json'
JBOOT_PATH_B = f'results/bootstrap_judge_{TAG_B}_{ARM}_{SPLIT}.json'
FLOOR_PATH_B = f'results/judge_floor_{TAG_B}_{ARM}_{SPLIT}.json'
AGREE_PATH_B = f'results/judge_agreement_{TAG_B}_{ARM}_{SPLIT}.json'

# The declared 0.058 floor is the primary rater's. This rater's floor is measured on the
# closest committed analogue: one prompting arm against another over the same 1,323 segments.
FLOOR_PAIR_B = ('afsp_full', BASELINE)

PRIOR_JUDGE_B = json.loads(Path(JUDGE_RESULTS_B).read_text(encoding='utf-8'))
PRIOR_USAGE_B = json.loads(Path(JUDGE_USAGE_B).read_text(encoding='utf-8'))
PRIOR_SPEND_B, PRIOR_CALLS_B = (PRIOR_USAGE_B['cumulative'][k] for k in ('cost_usd', 'calls'))

assert BASELINE in PRIOR_JUDGE_B, f'{BASELINE} is unjudged by {TAG_B}; there is no contrast'
assert all(c in PRIOR_JUDGE_B for c in FLOOR_PAIR_B), f'{FLOOR_PAIR_B} cannot price the floor'
BUY_B = [c for c in CONDS if c not in PRIOR_JUDGE_B]
for cond in CONDS:
    if cond in PRIOR_JUDGE_B:
        print(f"{cond} already scored by {TAG_B} "
              f"(Phi {PRIOR_JUDGE_B[cond]['mean']:.4f}); it is not re-bought")

# The batch rate is stable, so the whole ledger prices it better than one session does.
CUM_B = PRIOR_USAGE_B['cumulative']
RATE_B = CUM_B['cost_usd'] / CUM_B['calls']
N_CALLS_B = len(VAL) * len(BUY_B)
PROJECTED_B = RATE_B * N_CALLS_B

print(f"\n{PRIOR_USAGE_B['model']} over {PRIOR_USAGE_B['transport']} transport at "
      f"{PRIOR_USAGE_B['batch_discount']:.0%} of list")
print(f"  ${RATE_B * 1000:.3f} per 1000, measured over {CUM_B['calls']} calls")
print(f"{N_CALLS_B} calls for {', '.join(BUY_B) or 'nothing left'} project to ${PROJECTED_B:.2f}")
print(f'cumulative {TAG_B} spend to date ${PRIOR_SPEND_B:.2f}')

knn_fewshot already scored by gpt (Phi 3.6789); it is not re-bought
sparse_knn already scored by gpt (Phi 3.6985); it is not re-bought

gpt-5.6-terra over batch transport at 50% of list
  $0.659 per 1000, measured over 17199 calls
0 calls for nothing left project to $0.00
cumulative gpt spend to date $11.33


In [68]:
# Both raters must read the same frozen rubric or Phi_A and Phi_B are not comparable.
for path in (JUDGE_CFG, JUDGE_CFG_B):
    _c = yaml.safe_load(Path(path).read_text(encoding='utf-8'))
    assert _c['template_file'] == 'prompts/judge_eval.txt', _c['template_file']
    print(f"{path:30s} {_c['judge']['model']:16s} tag={_c.get('tag') or '(none)'}")

FROZEN = json.loads(Path('prompts/hashes.json').read_text(encoding='utf-8'))['templates']
DIGEST = hashlib.sha256(Path('prompts/judge_eval.txt').read_bytes()).hexdigest()
assert DIGEST == FROZEN['judge_eval.txt']['sha256'], 'the evaluation rubric has drifted'
print(f'\nrubric verified {DIGEST[:16]}')

configs/judge_eval.yaml        claude-haiku-4-5 tag=(none)
configs/judge_eval_gpt.yaml    gpt-5.6-terra    tag=gpt

rubric verified ffd6dad41acb0512


In [69]:
# Left False so a top-to-bottom re-run cannot authorise itself.
SPEND_OK_B = True
BUDGET_B_USD = 1.20

assert PROJECTED_B <= BUDGET_B_USD, (
    f'projection ${PROJECTED_B:.2f} exceeds the ${BUDGET_B_USD:.2f} cap')
print(f'authorised {SPEND_OK_B}   cap ${BUDGET_B_USD:.2f}   projected ${PROJECTED_B:.2f}')

authorised True   cap $1.20   projected $0.00


In [70]:
# Re-running over a complete cache submits nothing; an in-flight batch is resumed, not re-bought.
if not BUY_B:
    print(f'nothing to buy: every condition already carries Phi_{TAG_B}')
else:
    assert SPEND_OK_B, 'set SPEND_OK_B = True in the cell above to authorise the batch'
    r = subprocess.run([PY, 'manage.py', 'judge_batch', '--conditions', *BUY_B,
                        '--split', SPLIT, '--config', JUDGE_CFG_B], check=False)
    assert r.returncode == 0, f'judge_batch exited {r.returncode}'

nothing to buy: every condition already carries Phi_gpt


In [71]:
JUDGE_B = json.loads(Path(JUDGE_RESULTS_B).read_text(encoding='utf-8'))
HAVE_PHI_B = ARM in JUDGE_B
if not HAVE_PHI_B:
    print(f'{ARM} carries no Phi_{TAG_B}; the rest of section 13 has nothing to read')
else:
    r = subprocess.run([PY, 'manage.py', 'judge_ci', '--split', SPLIT, '--conditions', *CONDS,
                        '--tag', TAG_B, '--n_resamples', str(N_BOOT), '--alpha', str(ALPHA),
                        '--seed', str(SEED), '--results_path', JUDGE_CI_PATH_B], check=False)
    assert r.returncode == 0, f'judge_ci exited {r.returncode}'


Judge register fidelity Phi by condition  (split=val, n=1323 segments, resamples=10000, seed=42)
judge: gpt-5.6-terra  [tag gpt]
Phi = mean 1-5 rubric rating against the authorized reference; higher is better.

rank  condition    class  n     Phi     ci95              sd     P(this rank)  modal rank  mean rank
----------------------------------------------------------------------------------------------------
1     sparse_knn   study  1300  3.6985  [3.6561, 3.7416]  0.783  0.828         1 (0.828)   1.17     
2     knn_fewshot  study  1308  3.6789  [3.6345, 3.7231]  0.815  0.828         2 (0.828)   1.83     

Score distribution over the rubric (share of parsed segments)
condition    coverage  =1     =2     =3     =4     =5   
--------------------------------------------------------
sparse_knn   0.9826    0.015  0.060  0.230  0.600  0.095
knn_fewshot  0.9887    0.018  0.072  0.219  0.595  0.096

Adjacent ranks, paired bootstrap on the shared resamples  (a - b, 95% CI)
comparison        

In [ ]:
# Both read stored segment scores, so they cost nothing and can be re-run at will.
if HAVE_PHI_B:
    r = subprocess.run([PY, 'manage.py', 'bootstrap', '--metric', 'judge', '--split', SPLIT,
                        '--judge_tag', TAG_B, '--conditions', *CONDS, '--baseline', BASELINE,
                        '--pairs', f'{ARM}:{BASELINE}', '--n_resamples', str(N_BOOT),
                        '--alpha', str(ALPHA), '--seed', str(SEED), '--out', JBOOT_PATH_B],
                       check=False)
    assert r.returncode == 0, f'{TAG_B} judge bootstrap exited {r.returncode}'
    assert Path(JBOOT_PATH_B).exists(), f'no {JBOOT_PATH_B}: {ARM} has no {TAG_B} scores'

    hi, lo = FLOOR_PAIR_B
    r = subprocess.run([PY, 'manage.py', 'bootstrap', '--metric', 'judge', '--split', SPLIT,
                        '--judge_tag', TAG_B, '--conditions', hi, lo, '--baseline', lo,
                        '--pairs', f'{hi}:{lo}', '--n_resamples', str(N_BOOT),
                        '--alpha', str(ALPHA), '--seed', str(SEED), '--out', FLOOR_PATH_B],
                       check=False)
    assert r.returncode == 0, f'{TAG_B} floor bootstrap exited {r.returncode}'

wrote results/bootstrap_judge_gpt_sparse_knn_val.json

judge paired bootstrap  (resamples=10000, split=val)
comparison                n     diff   ci95             p       sig
-------------------------------------------------------------------
sparse_knn - knn_fewshot  1287  0.024  [-0.016, 0.064]  0.2488     

* = 95% CI excludes 0 (difference significant at α=0.05)
wrote results/judge_floor_gpt_sparse_knn_val.json

judge paired bootstrap  (resamples=10000, split=val)
comparison               n     diff   ci95             p       sig
------------------------------------------------------------------
afsp_full - knn_fewshot  1296  0.029  [-0.016, 0.072]  0.1968     

* = 95% CI excludes 0 (difference significant at α=0.05)


In [ ]:
if HAVE_PHI_B:
    JCI_B = json.loads(Path(JUDGE_CI_PATH_B).read_text(encoding='utf-8'))
    JBOOT_B = json.loads(Path(JBOOT_PATH_B).read_text(encoding='utf-8'))

    # This rater leaves a verdict unparsed on 1-2% of segments, so coverage is held to a
    # floor rather than to 1.0 as under the primary judge.
    COVERAGE_MIN_B = 0.97
    lost_b = sorted(set(PRIOR_JUDGE_B) - set(JUDGE_B))
    assert not lost_b, f'lost from {JUDGE_RESULTS_B}: {lost_b}'
    for cond in CONDS:
        assert JUDGE_B[cond]['model'] == JUDGE_B[BASELINE]['model'], f'{cond}: two raters, not one'
        assert JUDGE_B[cond]['sources'] == JUDGE_B[BASELINE]['sources'], f'{cond} is not paired'
        assert JUDGE_B[cond]['coverage'] >= COVERAGE_MIN_B, (
            f"{cond} coverage {JUDGE_B[cond]['coverage']:.4f} under {COVERAGE_MIN_B}")

    _f = json.loads(Path(FLOOR_PATH_B).read_text(encoding='utf-8'))['comparisons'][0]
    FLOOR_B = (_f['ci_high'] - _f['ci_low']) / 2
    PAIR_B = next(rec for rec in JBOOT_B['comparisons']
                  if (rec['a'], rec['b']) == (ARM, BASELINE))

    print(f"Phi_{TAG_B}   {JUDGE_B[BASELINE]['model']}")
    for cond in CONDS:
        p_lo, p_hi = JCI_B['cells'][cond]['phi_ci']
        print(f"  {cond:14s} Phi {JCI_B['cells'][cond]['phi']:.4f} [{p_lo:.4f}, {p_hi:.4f}]"
              f"  coverage {JUDGE_B[cond]['coverage']:.4f}")
    print(f"  {ARM} - {BASELINE}  {PAIR_B['diff']:+.4f} "
          f"[{PAIR_B['ci_low']:+.4f}, {PAIR_B['ci_high']:+.4f}] p={PAIR_B['p_value']:.4f} "
          f"n={PAIR_B['n']}")
    print(f"  floor {FLOOR_B:.4f}, measured on {FLOOR_PAIR_B[0]} - {FLOOR_PAIR_B[1]} "
          f'under this rater')

Phi_gpt   gpt-5.6-terra
  knn_fewshot    Phi 3.6789 [3.6345, 3.7231]  coverage 0.9887
  sparse_knn     Phi 3.6985 [3.6561, 3.7416]  coverage 0.9826
  sparse_knn - knn_fewshot  +0.0241 [-0.0155, +0.0645] p=0.2488 n=1287
  floor 0.0440, measured on afsp_full - knn_fewshot under this rater


In [ ]:
if HAVE_PHI_B:
    SCORES_B, SOURCES_B = _load_segment_scores('judge', CONDS, OUT, SPLIT, TAG_B)
    for cond in CONDS:
        assert SOURCES_B[cond] == SRC, f'{TAG_B}/{cond}: segment order is not the eval order'

    print(f'judge_{TAG_B}   {ARM} - {BASELINE}')
    for label, idx in STRATA.items():
        d = paired_bootstrap([SCORES_B[ARM][i] for i in idx],
                             [SCORES_B[BASELINE][i] for i in idx],
                             n_resamples=N_BOOT, alpha=ALPHA, seed=SEED)
        f = FLOOR_B * (len(VAL) / d['n']) ** 0.5
        print(f"  {label:26s} n={d['n']:5d}  {d['diff']:+.4f} "
              f"[{d['ci_low']:+.4f}, {d['ci_high']:+.4f}]  p={d['p_value']:.4f} "
              f"{'*' if d['significant'] else ' '}  floor {f:.4f}"
              f"{'' if abs(d['diff']) >= f else '  (under)'}")

    # Whether the two raters read the arm the same way is the reason for buying the second.
    if HAVE_PHI:
        print(f"\n  {'rater':6s} {'dPhi':>9s} {'floor':>8s} {'p':>8s}  separates")
        a_all = delta('judge', STRATA['all segments'])
        for name, rec, fl in (('A', a_all, FLOOR['judge']), (TAG_B, PAIR_B, FLOOR_B)):
            sep = rec['significant'] and abs(rec['diff']) >= fl
            print(f"  {name:6s} {rec['diff']:+9.4f} {fl:8.4f} {rec['p_value']:8.4f}  "
                  f"{'yes' if sep else 'no'}")

judge_gpt   sparse_knn - knn_fewshot
  full dose (n_sparse = 4)   n=  476  +0.0357 [-0.0252, +0.0945]  p=0.2670    floor 0.0733  (under)
  all routed                 n= 1175  +0.0289 [-0.0145, +0.0707]  p=0.1938    floor 0.0467  (under)
  all segments               n= 1287  +0.0241 [-0.0155, +0.0645]  p=0.2488    floor 0.0446  (under)

  rater       dPhi    floor        p  separates
  A        +0.0658   0.0580   0.0042  yes
  gpt      +0.0241   0.0440   0.2488  no


In [ ]:
# Reads both raters' segment caches, so it needs the primary pass complete, not just the pilot.
if HAVE_PHI_B and HAVE_PHI:
    r = subprocess.run([PY, 'manage.py', 'judge_agreement', '--split', SPLIT,
                        '--conditions', *CONDS, '--tag_b', TAG_B,
                        '--n_resamples', str(N_BOOT), '--alpha', str(ALPHA),
                        '--seed', str(SEED), '--results_path', AGREE_PATH_B], check=False)
    assert r.returncode == 0, f'judge_agreement exited {r.returncode}'
else:
    print(f'agreement needs both raters on {ARM}: '
          f'Phi_A {"present" if HAVE_PHI else "missing"}, '
          f'Phi_{TAG_B} {"present" if HAVE_PHI_B else "missing"}')


Judge-judge agreement  (split=val, resamples=10000, seed=42, 95% percentile CIs)
  judge A: claude-haiku-4-5  [tag (none)]
  judge B: gpt-5.6-terra  [tag gpt]
  same frozen rubric verified by digest: True

Coverage (segments parsed by each rater)
condition         n_total  n_a   n_b   n_both
---------------------------------------------
knn_fewshot       1323     1322  1308  1307  
sparse_knn        1323     1323  1300  1300  
commercial_haiku  1323     1323  1308  1308  

Rater agreement -- study_only
condition    n     Phi_A  Phi_B  A-B     ci95              qwk     qwk_ci            rho     exact  adj  
---------------------------------------------------------------------------------------------------------
knn_fewshot  1307  2.748  3.678  -0.930  [-0.975, -0.885]  +0.359  [+0.323, +0.394]  +0.562  28.8%  77.4%
sparse_knn   1300  2.814  3.698  -0.885  [-0.928, -0.842]  +0.367  [+0.329, +0.404]  +0.557  30.0%  80.7%
POOLED       2607  2.781  3.688  -0.907  [-0.938, -0.876]  +0.363  

In [ ]:
usage_b = json.loads(Path(JUDGE_USAGE_B).read_text(encoding='utf-8'))
assert usage_b['priced'], 'the second rater has no pricing table; cost_usd is a floor, not a bill'
SPENT_B = usage_b['cumulative']['cost_usd'] - PRIOR_SPEND_B
PAID_CALLS_B = usage_b['cumulative']['calls'] - PRIOR_CALLS_B
assert SPENT_B <= BUDGET_B_USD, f'${SPENT_B:.2f} spent against a ${BUDGET_B_USD:.2f} cap'

print(f"{PAID_CALLS_B} paid calls, ${SPENT_B:.2f} on {usage_b['model']} this session; "
      f"cumulative ${usage_b['cumulative']['cost_usd']:.2f}")
print(f'both raters, this pass ${SPENT + SPENT_B:.2f} against '
      f'${BUDGET_USD + BUDGET_B_USD:.2f} authorised across the two caps')

1323 paid calls, $0.88 on gpt-5.6-terra this session; cumulative $11.33
both raters, this pass $0.88 against $3.00 authorised across the two caps
